# Week 12 Final Project — AI-Powered Last-Mile Delivery Exception Handling Automation

Colab-ready full-code POC. Credentials are read only from Google Colab Secrets. All model calls use direct Bearer-authenticated HTTP requests to the Great Learning OpenAI-compatible endpoint; structured JSON is validated locally with Pydantic.


In [ ]:
!pip -q install "langgraph>=0.2" "sentence-transformers>=3.0" "pydantic>=2"
import os,re,json,time,sqlite3,subprocess,requests,numpy as np,pandas as pd
from pathlib import Path
from typing import TypedDict
from pydantic import BaseModel,Field
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph,START,END
from IPython.display import display
REPO="https://github.com/mschemerii/UT-Austin.git"; BRANCH="week-12-final-project"; ROOT=Path("/content/UT-Austin"); PROJECT=ROOT/"week-12-final-project"
def git(a,cwd=None,check=True):
 p=subprocess.run(["git",*a],cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
 if check and p.returncode: raise RuntimeError(p.stdout)
 return p.stdout.strip()
if not (ROOT/".git").exists(): print(git(["clone","--branch",BRANCH,"--single-branch",REPO,str(ROOT)]))
else:
 git(["fetch","origin",BRANCH],ROOT); git(["checkout",BRANCH],ROOT); print(git(["pull","--ff-only","origin",BRANCH],ROOT,False))
os.chdir(PROJECT)
REQ=["data/delivery_logs.csv","data/ground_truth.csv","data/customers.csv","data/lockers.csv","data/exception_resolution_playbook.txt"]
missing=[p for p in REQ if not Path(p).exists()]
if missing: raise FileNotFoundError("Missing: "+", ".join(missing))
def secret(n):
 try:
  from google.colab import userdata
  v=userdata.get(n)
 except Exception: v=os.environ.get(n)
 return str(v).strip() if v else None
KEY=secret("OPENAI_API_KEY"); BASE=secret("OPENAI_API_BASE") or secret("OPENAI_BASE_URL")
if not KEY or not BASE: raise ValueError("Enable Colab Secrets OPENAI_API_KEY and OPENAI_API_BASE.")
BASE=BASE.rstrip("/"); MODEL="gpt-4o-mini"; SESSION=requests.Session()
def chat(msgs):
 h={"Authorization":"Bearer "+(secret("OPENAI_API_KEY") or ""),"Content-Type":"application/json"}
 r=SESSION.post(BASE+"/chat/completions",headers=h,json={"model":MODEL,"messages":msgs,"temperature":0},timeout=90)
 if r.status_code==401:
  h={"Authorization":"Bearer "+(secret("OPENAI_API_KEY") or ""),"Content-Type":"application/json"}
  r=SESSION.post(BASE+"/chat/completions",headers=h,json={"model":MODEL,"messages":msgs,"temperature":0},timeout=90)
 if not r.ok: raise RuntimeError(f"LLM HTTP {r.status_code}: {r.text[:1000]}")
 return r.json()["choices"][0]["message"]["content"]
def parse_json(t):
 t=re.sub(r"^```(?:json)?\s*|\s*```$","",str(t).strip(),flags=re.I); a,b=t.find("{"),t.rfind("}")
 if a<0 or b<a: raise ValueError("No JSON object in model response")
 return json.loads(t[a:b+1])
def structured(schema,msgs):
 p=[{"role":"system","content":"Return ONLY valid JSON. Schema: "+json.dumps(schema.model_json_schema(),separators=(",",":"))},*msgs]
 for i in range(2):
  raw=chat(p)
  try:return schema.model_validate(parse_json(raw))
  except Exception as e:
   if i: raise
   p += [{"role":"assistant","content":str(raw)},{"role":"user","content":f"Correct the JSON. Error: {e}. JSON only."}]
print("Branch:",git(["branch","--show-current"],ROOT)); print("API test:",chat([{"role":"user","content":"Reply exactly: API connection successful"}]).strip())


In [ ]:
logs=pd.read_csv("data/delivery_logs.csv"); gt=pd.read_csv("data/ground_truth.csv")
customers=pd.read_csv("data/customers.csv",dtype=str,keep_default_na=False); lockers_df=pd.read_csv("data/lockers.csv",dtype=str,keep_default_na=False)
assert logs.shipment_id.nunique()==10 and gt.shipment_id.nunique()==10
conn=sqlite3.connect(":memory:"); customers.to_sql("customers",conn,index=False,if_exists="replace"); lockers_df.to_sql("lockers",conn,index=False,if_exists="replace")
raw=Path("data/exception_resolution_playbook.txt").read_text(); ms=list(re.finditer(r"^=== PAGE (\d+) ===\s*$",raw,re.M)); chunks=[]
for i,m in enumerate(ms):
 e=ms[i+1].start() if i+1<len(ms) else len(raw); t=re.sub(r"\s+"," ",raw[m.end():e]).strip(); s=0
 while s<len(t):
  z=min(len(t),s+900); chunks.append({"page":int(m.group(1)),"content":t[s:z]})
  if z==len(t): break
  s=max(s+1,z-150)
emb=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"); vec=emb.encode([x["content"] for x in chunks],normalize_embeddings=True,show_progress_bar=False)
BAD=["ignore previous","ignore all","system prompt","developer message","jailbreak","override instructions","do not follow"]
def bad(x): return any(p in str(x).lower() for p in BAD)
def consolidate(rs):
 d=pd.DataFrame(rs); d=d[~d.is_duplicate_scan.astype(str).str.lower().isin({"true","1","yes"})].sort_values("timestamp")
 if d.empty: raise ValueError("No usable event after deduplication")
 return d.iloc[-1].to_dict()
def cust(cid):
 r=conn.execute("select customer_id,name,tier,preferred_channel,exceptions_last_90d,active_credit from customers where customer_id=?",(cid,)).fetchone()
 return dict(zip(["customer_id","name","tier","preferred_channel","exceptions_last_90d","active_credit"],r))
def lockers(z):
 rs=conn.execute("select locker_id,address,zip_code,capacity_status,operating_hours,max_package_size from lockers where zip_code=?",(str(z),)).fetchall()
 return [dict(zip(["locker_id","address","zip_code","capacity_status","operating_hours","max_package_size"],r)) for r in rs]
def policy(q,k=4):
 qv=emb.encode([q],normalize_embeddings=True,show_progress_bar=False)[0]; sc=vec@qv; ids=np.argsort(sc)[::-1][:k]
 return [{"content":chunks[i]["content"],"page":chunks[i]["page"]} for i in ids]
def triggers(e,c):
 o=[]; s=str(e.status_code if hasattr(e,"status_code") else e["status_code"]).upper(); p=str(e["package_type"]).upper(); a=int(e["attempt_number"]); x=str(e["status_description"]).lower(); tier=c["tier"].upper(); h=int(float(c["exceptions_last_90d"] or 0))
 if a>=3:o.append("AUTOMATIC: third failed delivery attempt")
 if tier=="VIP" and h>=3:o.append("AUTOMATIC: VIP with >=3 recent exceptions")
 if s=="DAMAGED" and p=="PERISHABLE":o.append("AUTOMATIC: damaged perishable")
 m=re.search(r"(\d+(?:\.\d+)?)\s*(?:hr|hour|hours)",x)
 if s=="WEATHER_DELAY" and p=="PERISHABLE" and m and float(m.group(1))>4:o.append("AUTOMATIC: perishable weather delay >4h")
 if tier=="STANDARD" and h>5:o.append("DISCRETIONARY: high exception history")
 return o
print(len(logs),"log rows;",len(chunks),"playbook chunks")


In [ ]:
class R(BaseModel): is_exception:bool; exception_type:str; resolution:str; rationale:str; escalate:bool
class M(BaseModel): tone_label:str; channel:str; message:str
class K(BaseModel): pass_check:bool; feedback:str
class C(BaseModel): score:int=Field(ge=1,le=5); explanation:str
class S(TypedDict,total=False):
 rows:list; event:dict; customer:dict; lockers:list; policy:list; triggers:list; resolution:dict; resolution_critic:dict; communication:dict; communication_critic:dict; tools:list; cites:list; resolution_retried:bool; communication_retried:bool; start:float; latency:float
RP="""Resolve only from evidence/playbook. Allowed resolution: N/A, RESCHEDULE, REROUTE_TO_LOCKER, REPLACE, RETURN_TO_SENDER. DELIVERED/routine SCANNED are not exceptions. Failed attempts normally RESCHEDULE. Third attempt must escalate and may use locker path. ADDRESS_ISSUE normally verify then RESCHEDULE. REFUSED -> RETURN_TO_SENDER. Severe damage, damaged perishables, and perishable weather delay > threshold -> REPLACE. FRAGILE damage has lower replacement threshold. Any supplied escalation trigger forces escalate=true."""
MP="""Draft concise customer communication matching approved resolution. Non-exception: tone_label=N/A and empty message. VIP/PREMIUM=FORMAL; STANDARD=CASUAL. Use preferred channel. Do not invent compensation, dates, locker details, or internal escalation."""
KP="""Validate candidate against event, customer, lockers, playbook, triggers and approved resolution. Reject contradictions, unsupported facts, invalid actions, wrong tone/channel, or missing required escalation."""
def prep(s):
 e=consolidate(s["rows"])
 if bad(e["status_description"]): raise ValueError("Prompt injection in driver note")
 c=cust(e["customer_id"]); l=lockers(str(e["delivery_address"]).split()[-1]); p=policy(f'{e["status_code"]} {e["package_type"]} attempt {e["attempt_number"]} locker escalation'); tr=triggers(e,c)
 return {"event":e,"customer":c,"lockers":l,"policy":p,"triggers":tr,"tools":["consolidate","customer_lookup","locker_lookup","playbook_search","escalation_rules"],"cites":[f'playbook p.{x["page"]}' for x in p]}
def ev(s): return {"event":s["event"],"customer":{k:v for k,v in s["customer"].items() if k!="name"},"lockers":s["lockers"],"playbook":s["policy"],"triggers":s["triggers"]}
def res(s):
 o=structured(R,[{"role":"system","content":RP},{"role":"user","content":json.dumps(ev(s),default=str)}])
 if s["triggers"]:o.escalate=True
 return {"resolution":o.model_dump(),"resolution_retried":False}
def rc(s):
 ctx={**ev(s),"candidate":s["resolution"]}; k=structured(K,[{"role":"system","content":KP},{"role":"user","content":json.dumps(ctx,default=str)}]); r=s["resolution"]; retry=False
 if not k.pass_check:
  o=structured(R,[{"role":"system","content":RP},{"role":"user","content":json.dumps({**ev(s),"previous":r,"critic_feedback":k.feedback},default=str)}])
  if s["triggers"]:o.escalate=True
  r=o.model_dump(); retry=True; k=structured(K,[{"role":"system","content":KP},{"role":"user","content":json.dumps({**ev(s),"candidate":r},default=str)}])
 return {"resolution":r,"resolution_critic":k.model_dump(),"resolution_retried":retry}
def msg(s): return {"communication":structured(M,[{"role":"system","content":MP},{"role":"user","content":json.dumps({"customer":s["customer"],"event":s["event"],"resolution":s["resolution"]},default=str)}]).model_dump(),"communication_retried":False}
def mc(s):
 ctx={"customer":s["customer"],"event":s["event"],"resolution":s["resolution"],"candidate":s["communication"]}; k=structured(K,[{"role":"system","content":KP},{"role":"user","content":json.dumps(ctx,default=str)}]); m=s["communication"]; retry=False
 if not k.pass_check:
  m=structured(M,[{"role":"system","content":MP},{"role":"user","content":json.dumps({**ctx,"critic_feedback":k.feedback},default=str)}]).model_dump(); retry=True; k=structured(K,[{"role":"system","content":KP},{"role":"user","content":json.dumps({**ctx,"candidate":m},default=str)}])
 return {"communication":m,"communication_critic":k.model_dump(),"communication_retried":retry}
def done(s): return {"latency":time.perf_counter()-s["start"]}
g=StateGraph(S)
for n,f in [("prep",prep),("res",res),("rc",rc),("msg",msg),("mc",mc),("done",done)]:g.add_node(n,f)
g.add_edge(START,"prep");g.add_edge("prep","res");g.add_edge("res","rc");g.add_edge("rc","msg");g.add_edge("msg","mc");g.add_edge("mc","done");g.add_edge("done",END);app=g.compile()
print(app.get_graph().draw_mermaid())


In [ ]:
def yn(v):return str(v).strip().upper() in {"YES","TRUE","1","Y"}
def norm(v):return str(v).strip().upper().replace(" ","_").replace("-","_")
truth=gt.groupby("shipment_id",as_index=False).first().set_index("shipment_id").to_dict("index"); groups={sid:d.to_dict("records") for sid,d in logs.groupby("shipment_id")}
def evaluate(sid,s,t):
 r=s["resolution"];m=s["communication"];cls=bool(r["is_exception"])==yn(t["is_exception"]);rr=norm(r["resolution"])==norm(t["expected_resolution"]);tone=norm(m["tone_label"])==norm(t["expected_tone"]); ee=None if str(t["should_escalate"]).strip().upper() in {"N/A","NA",""} else yn(t["should_escalate"]); esc=None if ee is None else bool(r["escalate"])==ee; tool={"consolidate","customer_lookup","locker_lookup","playbook_search","escalation_rules"}.issubset(set(s["tools"]));co=structured(C,[{"role":"system","content":"Score reasoning coherence 1-5."},{"role":"user","content":json.dumps({"event":s["event"],"triggers":s["triggers"],"resolution":r,"communication":m},default=str)}])
 return {"shipment_id":sid,"classification_correct":cls,"resolution_correct":rr,"tone_correct":tone,"task_complete":cls and rr and tone,"escalation_correct":esc,"tool_call_correct":tool,"coherence_score":co.score,"latency_sec":round(s["latency"],3),"pred_resolution":r["resolution"],"pred_escalate":r["escalate"],"pred_tone":m["tone_label"],"resolution_critic_pass":s["resolution_critic"]["pass_check"],"communication_critic_pass":s["communication_critic"]["pass_check"],"tool_trace":" → ".join(s["tools"]),"citations":", ".join(s["cites"])}
out=[];states={}
for sid in sorted(groups):
 print("Running",sid); s=app.invoke({"rows":groups[sid],"start":time.perf_counter()});states[sid]=s;out.append(evaluate(sid,s,truth[sid]))
results=pd.DataFrame(out);display(results);esc=results.escalation_correct.dropna()
metrics=pd.DataFrame({"metric":["Task completion rate","Escalation accuracy","Tool call accuracy","Mean coherence (1-5)","Mean latency (sec)"],"value":[results.task_complete.mean(),esc.mean(),results.tool_call_correct.mean(),results.coherence_score.mean(),results.latency_sec.mean()]});display(metrics)
display(results[(~results.task_complete)|(results.escalation_correct==False)|(~results.tool_call_correct)|(results.coherence_score<4)])


## Conclusions

Use the executed tables above as the quantitative evidence. The system keeps deterministic escalation rules outside the LLM, grounds decisions in playbook retrieval with page citations, validates outputs with critic agents, and exposes case-level failures. Retain human review for escalated or low-confidence cases before production use.
